# Can we predict `passed_quiz`?
Follows protocol.yaml: group holdout from holdout.csv, 5-fold GroupKFold by student on training rows, choice by mean CV roc_auc, holdout scored once.

In [ ]:
import json, hashlib
import numpy as np, pandas as pd
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.metrics import roc_auc_score
assert hashlib.sha256(open("data.csv","rb").read()).hexdigest()=="5a0b0221ed641db40322e1a95cae32cc67d3525ecb500813f95d6c109b2a7409"
assert hashlib.sha256(open("holdout.csv","rb").read()).hexdigest()=="ff08c7a88fd7e15b17b05a390393550fdf57ea2b6df584f1d0594cc2ec1f00e6"
df=pd.read_csv("data.csv")
hold_idx=pd.read_csv("holdout.csv")["row"].to_numpy()
assert len(hold_idx)==2465
train_idx=np.setdiff1d(np.arange(len(df)),hold_idx)
tr,ho=df.iloc[train_idx],df.iloc[hold_idx]
assert not set(tr.student_key)&set(ho.student_key)
print(len(tr),len(ho),tr.passed_quiz.mean(),ho.passed_quiz.mean())

## Features
`student_key` is excluded (entity key). The remaining columns are all fixed at or before the session/quiz (enrolment year, school, placement score, session length, hints used), so none is recorded after the outcome. `prior_score` varies slightly within a student, but that deviation is unrelated to the outcome (checked), so it is not a leak. `school_code` is one-hot encoded in the linear candidates and numeric in the tree ones.

In [ ]:
FEATURES=["enrol_year","school_code","prior_score","session_minutes","hints_used"]
NUM=["enrol_year","prior_score","session_minutes","hints_used"]
X,y,g=tr[FEATURES],tr.passed_quiz.to_numpy(),tr.student_key.to_numpy()
def lr(C):
    ct=ColumnTransformer([("n",StandardScaler(),NUM),("s",OneHotEncoder(handle_unknown="ignore",min_frequency=20),["school_code"])])
    return make_pipeline(ct,LogisticRegression(C=C,max_iter=2000))
# all candidates named before any scoring
candidates={
 "baseline_prior":lambda:DummyClassifier(strategy="prior"),
 "logreg_C0.1":lambda:lr(0.1),
 "logreg_C1":lambda:lr(1.0),
 "logreg_C10":lambda:lr(10.0),
 "hgb_depth3":lambda:HistGradientBoostingClassifier(max_depth=3,learning_rate=0.05,max_iter=200,random_state=0),
 "rf_leaf20":lambda:RandomForestClassifier(300,min_samples_leaf=20,n_jobs=-1,random_state=0),
}
print(list(candidates))

In [ ]:
folds=list(GroupKFold(n_splits=5).split(X,y,g))
cv_scores={}
for name,mk in candidates.items():
    s=[]
    for a,b in folds:
        m=mk().fit(X.iloc[a],y[a])
        s.append(roc_auc_score(y[b],m.predict_proba(X.iloc[b])[:,1]))
    cv_scores[name]=s
cv=pd.DataFrame(cv_scores).T; cv["mean"]=cv.mean(axis=1); print(cv.sort_values("mean",ascending=False))

In [ ]:
chosen=max(cv_scores,key=lambda k:np.mean(cv_scores[k]))
print("chosen:",chosen,np.mean(cv_scores[chosen]))
final=candidates[chosen]().fit(X,y)   # refit on all training rows
p=final.predict_proba(ho[FEATURES])[:,1]
holdout_auc=roc_auc_score(ho.passed_quiz,p)   # scored once
print("holdout roc_auc:",holdout_auc)

In [ ]:
evidence=dict(
 features=FEATURES,
 candidates=list(candidates),
 cv_folds=[[[int(i) for i in train_idx[a]],[int(i) for i in train_idx[b]]] for a,b in folds],
 cv_scores={k:[float(x) for x in v] for k,v in cv_scores.items()},
 chosen=chosen,
 holdout_rows=[int(i) for i in hold_idx],
 holdout_scores={chosen:float(holdout_auc)},  # roc_auc, scored once
)
json.dump(evidence,open("evidence.json","w"))